# Content segmentation

Executed analysis of the supplied historical catalog. Feature preparation, model fitting and selection are implemented in the task-owned Python package. This notebook reads its frozen outputs; it does not retune the model or use ratings as clustering labels.

The raw 8,790 rows become 8,783 title/type/year entities after removing seven repeated entity rows. Genre, country and type indicator blocks are individually L2-normalized. Year and separate log-duration fields use robust scaling and clipping. The full weighted feature space is used for fitting and internal scoring; PCA is visualization only.


In [1]:
from netflix_ml.tasks.task_04.service import load_segmentation

service = load_segmentation()
result = service.results
print('Run:', result['run_id'])
print('Catalog entities:', result['catalog_rows'])
print('Feature dimensions:', result['feature_count'])
print('Ratings are excluded from all fitting and naming.')


Run: 20261106T115043590368Z
Catalog entities: 8783
Feature dimensions: 133
Ratings are excluded from all fitting and naming.


In [2]:
print('Algorithm / k:', result['algorithm'], result['k'])
print('Mean silhouette:', result['selected']['silhouette_mean'])
print('Silhouette standard deviation:', result['selected']['silhouette_std'])
print('Seed stability ARI:', result['selected']['stability_ari'])
print('PCA 2D variance:', result['projection_variance'])
print('Scoring sample:', result['sample_size'])
print('Block amplitude weights:', result['feature_weights'])


Algorithm / k: kmeans 10
Mean silhouette: 0.24080575754813563
Silhouette standard deviation: 0.0034076856053579024
Seed stability ARI: 0.661990597885784
PCA 2D variance: 0.2585429860933723
Scoring sample: 1500
Block amplitude weights: {'type': 0.35, 'genre': 1.0, 'country': 0.35, 'year': 0.25, 'duration': 0.25}


In [3]:
import pandas as pd

comparison = pd.DataFrame(result['experiments'])
print(comparison[['algorithm','k','silhouette_mean','davies_bouldin','stability_ari','eligible']].to_string(index=False))


       algorithm  k  silhouette_mean  davies_bouldin  stability_ari  eligible
          kmeans  2         0.166194        2.266636       1.000000      True
          kmeans  3         0.149282        2.358443       1.000000      True
          kmeans  4         0.174509        2.253739       0.833007      True
          kmeans  5         0.186732        2.113783       0.796310      True
          kmeans  6         0.193925        2.137064       0.666338      True
          kmeans  7         0.210208        1.953365       0.624877      True
          kmeans  8         0.218044        1.871056       0.832742      True
          kmeans  9         0.226649        1.829471       0.559414      True
          kmeans 10         0.240806        1.734353       0.661991      True
minibatch_kmeans  2         0.156278        2.360117       0.567413      True
minibatch_kmeans  3         0.160324        2.302306       0.611675      True
minibatch_kmeans  4         0.168280        2.185254       0.688

In [4]:
for profile in result['profiles']:
    print(f"Cluster {profile['id']}: {profile['name']} ({profile['size']} titles)")
    print('Movie minutes median:',profile['median_movie_minutes'],'TV seasons median:',profile['median_tv_seasons'])
    print('Top countries:',profile['countries'])
    print('Representatives:',[row['title'] for row in profile['representatives']])


Cluster 0: Stand-Up Comedy / Music & Musicals (343 titles)
Movie minutes median: 65.0 TV seasons median: None
Top countries: [{'name': 'United States', 'count': 223}, {'name': 'United Kingdom', 'count': 22}, {'name': 'Mexico', 'count': 18}, {'name': 'Not Given', 'count': 13}, {'name': 'Brazil', 'count': 9}]
Representatives: ['Patton Oswalt: Talking for Clapping', 'Theo Von: No Offense', 'Hannibal Buress: Comedy Camisado', 'Joe Rogan: Triggered', 'Dana Carvey: Straight White Male, 60']
Cluster 1: Dramas / International Movies (1914 titles)
Movie minutes median: 108.5 TV seasons median: None
Top countries: [{'name': 'United States', 'count': 522}, {'name': 'India', 'count': 503}, {'name': 'United Kingdom', 'count': 117}, {'name': 'Not Given', 'count': 61}, {'name': 'France', 'count': 60}]
Representatives: ['Doctor Bello', 'Killa', 'Vitti Dandu', 'Rahasya', 'Ringan']
Cluster 2: International TV Shows / Docuseries (1359 titles)
Movie minutes median: None TV seasons median: 1.0
Top countrie

## Interpretation

K-Means with 10 clusters wins within the declared range 2–10. Mean silhouette around 0.24 indicates partial separation, not a strong ground-truth partition. The winner is at the upper search boundary, so the experiment does not establish a globally optimal k. Moderate seed stability also shows that some boundaries change with initialization.

Groups describe genre/type patterns such as documentaries, children’s content, drama and TV content. Their names summarize top genres, not verified audience or popularity segments. Representatives are nearest to the group mean in full feature space. Country counts are multi-label; missing country is an explicit source category.

The two PCA components capture only about a quarter of feature variance. Visual overlap must not be mistaken for clustering failure or success. Internal scores and same-catalog seed stability do not demonstrate future-release generalization, human semantic correctness, or official rating behavior.


## Exported report figure

![cluster_projection.png](../../reports/task-04/figures/cluster_projection.png)

Generated by the recorded clustering run; displayed here without rerunning selection.

## Exported report figure

![model_comparison.png](../../reports/task-04/figures/model_comparison.png)

Generated by the recorded clustering run; displayed here without rerunning selection.